In [1]:
import sys
from pathlib import Path
from coval.eval.evaluator import (
    muc,
    b_cubed,
    ceafe,
    lea
)
from collections import defaultdict


PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)
from src.evaluation import evaluate_batch, evaluate_attribute

c:\Users\zakga\OneDrive\Documents\code\LeREaD


c:\Users\zakga\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def build_mention_to_gold_index(clusters):
    """mention -> cluster index (int), needed for muc/b_cubed/lea"""
    m2c = {}
    for i, cluster in enumerate(clusters):
        for m in cluster:
            m2c[m] = i
    return m2c


def evaluate_metric(metric, pred_clusters, gold_clusters,
                     mention_to_gold, mention_to_pred):
    if metric is ceafe:
        p_num, p_den, r_num, r_den = metric(pred_clusters, gold_clusters)
    elif metric is lea:
        p_num, p_den = metric(pred_clusters, gold_clusters, mention_to_gold)
        r_num, r_den = metric(gold_clusters, pred_clusters, mention_to_pred)
    else:  # muc, b_cubed
        p_num, p_den = metric(pred_clusters, mention_to_gold)
        r_num, r_den = metric(gold_clusters, mention_to_pred)

    precision = p_num / p_den if p_den else 0.0
    recall = r_num / r_den if r_den else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) else 0.0)

    return precision, recall, f1


def evaluate_coref(gold_clusters, pred_clusters):
    gold_m2g = build_mention_to_gold_index(gold_clusters)
    pred_m2g = build_mention_to_gold_index(pred_clusters)

    results = {}
    for name, metric in [("MUC", muc), ("B3", b_cubed),
                          ("CEAF_e", ceafe), ("LEA", lea)]:
        p, r, f1 = evaluate_metric(
            metric, pred_clusters, gold_clusters, gold_m2g, pred_m2g
        )
        results[name] = {"precision": p, "recall": r, "f1": f1}

    return results


def print_evaluation(scores):
    print("=== Coreference Evaluation ===\n")

    for name in ["MUC", "B3", "CEAF_e", "LEA"]:
        s = scores[name]
        print(f"{name}:")
        print(f"  Precision: {s['precision']:.4f}")
        print(f"  Recall:    {s['recall']:.4f}")
        print(f"  F1:        {s['f1']:.4f}\n")

    conll = (scores["MUC"]["f1"] + scores["B3"]["f1"] + scores["CEAF_e"]["f1"]) / 3
    print(f"=== CoNLL average F1: {conll:.4f} ===")


In [13]:
import json
ground_truth_file = "../output/ground_truth_clusters_2002.json"
system_output_file = "../output/system_clusters_2002.json"

with open(ground_truth_file, "r") as f:
    ground_truth_clusters = json.load(f)

with open(system_output_file, "r") as f:
    system_clusters = json.load(f)

In [10]:
print(ground_truth_clusters)

[['1'], ['4', '17', '20', '23', '28', '92', '115', '144'], ['6', '10', '15', '35', '59', '71', '78', '80', '82', '84', '86', '88', '90', '95', '97', '99', '101', '103', '109', '111', '113', '123', '256'], ['12', '33', '46', '48', '51', '53', '55', '57', '63', '65', '69', '73', '76', '118'], ['31', '173', '178', '179'], ['37', '40'], ['42'], ['61', '229', '233', '235'], ['67', '224', '227', '237'], ['105', '262'], ['107', '259'], ['120', '240', '244', '246', '248', '250', '252', '254'], ['125'], ['130', '134'], ['136'], ['140', '168', '172', '201'], ['146'], ['150'], ['154'], ['158'], ['164'], ['180'], ['184'], ['188'], ['192'], ['196'], ['204', '207'], ['209'], ['214'], ['219'], ['222'], ['265', '270']]


In [11]:
print(system_clusters)

[['1', '134', '172'], ['4', '17', '20', '23', '28', '92', '115', '144'], ['6', '10', '15', '35', '59', '71', '78', '80', '82', '84', '86', '88', '90', '95', '97', '99', '101', '103', '109', '111', '113', '256'], ['12', '33', '46', '48', '51', '53', '55', '57', '63', '65', '69', '73', '76', '118', '227'], ['31', '173'], ['37', '40'], ['42'], ['61', '229', '233', '235'], ['67', '224', '237'], ['105', '262'], ['107', '259'], ['120', '204', '240', '244', '246', '248', '250', '252', '254'], ['123'], ['125'], ['130'], ['136'], ['140', '168', '201'], ['146'], ['150'], ['154'], ['158'], ['164', '178', '179'], ['180'], ['184'], ['188'], ['192'], ['196', '207'], ['209'], ['214'], ['219'], ['222'], ['265', '270']]


In [14]:
scores = evaluate_coref(ground_truth_clusters, system_clusters)
print_evaluation(scores)

for metric, value in scores.items():
    print(f"{metric}: {value}")

=== Coreference Evaluation ===

MUC:
  Precision: 0.8842
  Recall:    0.8876
  F1:        0.8859

B3:
  Precision: 0.7926
  Recall:    0.8117
  F1:        0.8021

CEAF_e:
  Precision: 0.8622
  Recall:    0.8499
  F1:        0.8560

LEA:
  Precision: 0.7549
  Recall:    0.7706
  F1:        0.7627

=== CoNLL average F1: 0.8480 ===
MUC: {'precision': 0.8841698841698842, 'recall': 0.8875968992248062, 'f1': 0.8858800773694392}
B3: {'precision': 0.7926097868933233, 'recall': 0.8117182952548805, 'f1': 0.8020502441843427}
CEAF_e: {'precision': np.float64(0.8622254179831877), 'recall': np.float64(0.8499079120119993), 'f1': np.float64(0.8560223574221576)}
LEA: {'precision': 0.7549482191250484, 'recall': 0.7706005173688101, 'f1': 0.7626940710940259}


In [43]:
def muc_prf(score):
    num, den = score
    f1 = num / den if den else 0
    return f1

def b3_prf(score):
    val, n = score
    return val / 100 if val > 1 else val

def ceafe_prf(score):
    # (p_num, p_den, r_num, r_den)
    p_num, p_den, r_num, r_den = score

    p = p_num / p_den if p_den else 0
    r = r_num / r_den if r_den else 0
    f1 = 2 * p * r / (p + r) if (p + r) else 0

    return p, r, f1

def lea_prf(score):
    val, n = score
    return val / 100 if val > 1 else val

In [44]:
def print_evaluation(scores):

    print("=== Coreference Evaluation ===\n")

    # MUC
    muc_f1 = muc_prf(scores["MUC"])
    print(f"MUC:\n  F1: {muc_f1:.4f}\n")

    # B3
    b3 = b3_prf(scores["B3"])
    print(f"B3:\n  F1: {b3:.4f}\n")

    # CEAF_e
    p, r, f1 = ceafe_prf(scores["CEAF_e"])
    print(f"CEAF_e:\n  Precision: {p:.4f}\n  Recall: {r:.4f}\n  F1: {f1:.4f}\n")

    # LEA
    lea = lea_prf(scores["LEA"])
    print(f"LEA:\n  F1: {lea:.4f}\n")

    conll = (muc_f1 + b3 + f1) / 3
    print(f"=== CoNLL average F1: {conll:.4f} ===")

In [45]:
print_evaluation(scores)

=== Coreference Evaluation ===

MUC:
  F1: 0.9299

B3:
  F1: 2.6997

CEAF_e:
  Precision: 0.7829
  Recall: 0.8149
  F1: 0.7986

LEA:
  F1: 3.1600

=== CoNLL average F1: 1.4761 ===
